# Notebook 02 - Curated ML EDA (Wave B)

Phan tich MOT `dataset_version` da `PASS`, dau vao **tuong minh** (database, batch, dataset_version, thu muc artifact) - khong co pointer, khong "dataset moi nhat".
Chay qua `python run_wave_b.py --database ... --batch-id ... --dataset-version ... --dataset-dir ...` (khong mo truc tiep).

Quy tac: moi SQL nam trong `src/wave_b_queries.py` (query catalog co SHA-256), moi tinh toan trong `src/wave_b.py`, ghi output trong `src/wave_b_output.py`;
notebook chi dieu phoi. Mau so luon tach bat (`n_*`); mau so 0 => ty le null. Kiem tra toan ven bat buoc (= 0): neu vi pham, analysis bi danh dau FAILED.
Neu dataset co purpose != official thi day la phan tich **rehearsal/exploratory**, khong dung de ket luan danh gia model.

In [ ]:
import os
import sys
from pathlib import Path

try:
    SRC_DIR = Path(os.environ["EDA_SRC_DIR"])
    ANALYSIS_DIR = Path(os.environ["EDA_ANALYSIS_DIR"])
    DATABASE = os.environ["EDA_WB_DATABASE"]
    BATCH_ID = os.environ["EDA_WB_BATCH_ID"]
    DATASET_VERSION = os.environ["EDA_WB_DATASET_VERSION"]
    DATASET_DIR = Path(os.environ["EDA_WB_DATASET_DIR"])
except KeyError as exc:
    raise RuntimeError("Thieu bien moi truong Wave B - chay qua run_wave_b.py, khong mo truc tiep trong Jupyter.") from exc
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import wave_b
import wave_b_inputs
import wave_b_output

## 1. Preflight (fail-closed)

In [ ]:
inputs = wave_b_inputs.WaveBInputs(DATABASE, BATCH_ID, DATASET_VERSION, DATASET_DIR)
conn_ctx = wave_b_inputs.connect_explicit(DATABASE, BATCH_ID)
conn, snapshot = conn_ctx.__enter__()
try:
    preflight = wave_b_inputs.preflight(conn, inputs)
    print("PREFLIGHT OK:", DATASET_VERSION, "| generation:", preflight["artifact_generation"])
    print("ml_table_rows:", preflight["ml_table_rows"], "| selected_samples:", preflight["selected_samples"])
finally:
    pass

## 2. Nap du lieu, tinh bang, hinh, ghi output, kiem toan ven

In [ ]:
try:
    summary = wave_b_output.run_all(conn, inputs, preflight, ANALYSIS_DIR)
finally:
    conn_ctx.__exit__(None, None, None)
print("hard_violations_total =", summary["hard_violations_total"])
summary["usable_labels_by_horizon"], summary["usable_rate_of_source_eligible_by_horizon"]

## 3. Coverage matrix yeu cau muc 8 va bang chinh

In [ ]:
import pandas as pd
coverage = pd.read_csv(ANALYSIS_DIR / "coverage_matrix_b.csv")
assert coverage["covered"].all(), "co yeu cau muc 8 khong co bang"
coverage[["bullet", "requirement", "tables", "figures"]]

In [ ]:
pd.read_csv(ANALYSIS_DIR / "tables" / "b05_label_rates.csv")

In [ ]:
pd.read_csv(ANALYSIS_DIR / "tables" / "b08_coverage_attrition.csv")

In [ ]:
pd.read_csv(ANALYSIS_DIR / "tables" / "b02_match_by_phase.csv")

## 4. Dien giai - chi trong gioi han bang chung

- Cap ly thuyet o `b08` la noi tai dataset; khong cung quan the voi `theoretical_date_pairs` cua Wave A.
- `b11` (review tier) la phan khuc post-hoc tren snapshot luc build warehouse.
- `b12` (exact/alias) la audit/sensitivity; trang thai khop cua target la thong tin tuong lai, khong phai feature.
- Dataset rehearsal chi co khoang 45 ngay quan sat: moi horizon deu exploratory; khong rut ket luan model tu day.